In [1]:
# Imports
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier, early_stopping, log_evaluation                        
from sklearn.metrics import log_loss 
from sklearn.model_selection import train_test_split
import duckdb
import joblib
from itertools import product

In [2]:
# Define different options for features
# Base features: Game scenario + plate location
base = ["balls", "strikes", "outs_when_up", "p_throws", "stand", "on_1b", "on_2b", "on_3b", "plate_x", "z_diff"]

# Base + Pitch Metrics
base_w_pitch = base + ["release_speed", "release_pos_x", "release_pos_z", "pfx_x", "pfx_z", "release_spin_rate", "release_extension"]

target = "swing"

# Load Training Data

In [3]:
# Use DuckDB to read SQL queries
train_query = """
    SELECT
        *
    FROM
        data.parquet
    WHERE
        game_year < 2026
"""
df = duckdb.sql(train_query).df()

In [4]:
# Drop rows with NAs
df1 = df.dropna(subset=base_w_pitch)

In [5]:
len(df1)

3815918

# Create Function To Build Model

In [6]:
# Function that builds model based on iteration of hyperparamters
# Evaluates with validation set, which optimizes for log loss
def build_model(train_df, val_df, features, target, learning_rate, max_depth, num_leaves, min_child_samples):
    
    model = LGBMClassifier(
        n_estimators=10000,
        learning_rate=learning_rate,
        max_depth=max_depth,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
    
    model.fit(
        train_df[features], train_df[target],
        eval_set=[(val_df[features], val_df[target])],
        eval_metric="binary_logloss",
        callbacks=[early_stopping(stopping_rounds=50, verbose=False), log_evaluation(period=0)]
    )
    
    return model, model.best_iteration_, model.best_score_["valid_0"]["binary_logloss"]

In [7]:
# Define choices for hyperparameters and features (excluding learning rate, will define later)
md = [4, 6, 8, -1]
nl = [31, 63, 127, 255]
mcs = [100, 250, 500, 1000, 2000]
feats = {"base": base, "base_w_pitch": base_w_pitch}

In [8]:
# Create training and validation set
# 90% of data for training, 10% for validation
train_df, val_df = train_test_split(df1, test_size=0.1, random_state=42)

In [9]:
# Grid - all combinations of defined features/parameters
# Ensures num_leaves and max_depth are within reasonable correlation
grid = [
    (fname, d, l, m)
    for fname, d, l, m in product(feats, md, nl, mcs)
    if d == -1 or l <= 0.65 * 2**d
]

# Stage 1

#### Using a set learning rate (0.1), find the best performing combinations of features and parameters

In [10]:
stage1 = []
# Loop through each feature/parameters combo
for fname, d, l, m in grid:

    # Attain optimal # of boosting rounds and MSE score from this iteration
    _, n_trees, score = build_model(train_df, val_df, feats[fname], target, 0.1, d, l, m)

    # Create row with parameters and score
    stage1.append({"features": fname, 
                   "learning_rate": 0.1, 
                   "max_depth": d, 
                   "num_leaves": l,
                   "min_child_samples": m, 
                   "n_trees": n_trees, 
                   "val_logloss": score})

# Create df of all iterations
stage1_df = pd.DataFrame(stage1).sort_values("val_logloss")

In [11]:
stage1_df

,features,learning_rate,max_depth,num_leaves,min_child_samples,n_trees,val_logloss
73,base_w_pitch,0.1,-1,127,1000,773,0.459656
74,base_w_pitch,0.1,-1,127,2000,776,0.459746
64,base_w_pitch,0.1,-1,31,2000,1850,0.459764
54,base_w_pitch,0.1,8,63,2000,1097,0.459806
79,base_w_pitch,0.1,-1,255,2000,392,0.459811
...,...,...,...,...,...,...,...
32,base,0.1,-1,127,500,154,0.504133
30,base,0.1,-1,127,100,115,0.504148
36,base,0.1,-1,255,250,97,0.504151
39,base,0.1,-1,255,2000,96,0.504162


# Stage 2

#### Try the best parameters with lower learning rates to see if mse scores improve

In [13]:
# 5 Best performing parameters
top = stage1_df.head(5)

stage2 = []
# Top 5 parameter combos
for _, row in top.iterrows():
    
    # New learning rates to try
    for lr in [0.01, 0.02, 0.05]:

        # Attain optimal # of boosting rounds and MSE score from this iteration
        _, n_trees, score = build_model(train_df, val_df, feats[row["features"]], target, lr, int(row["max_depth"]), int(row["num_leaves"]), int(row["min_child_samples"])
        )

        # Create row with parameters and score
        stage2.append({"features": row["features"], 
                       "learning_rate": lr,
                       "max_depth": int(row["max_depth"]), 
                       "num_leaves": int(row["num_leaves"]),
                       "min_child_samples": int(row["min_child_samples"]),
                       "n_trees": n_trees, 
                       "val_logloss": score})

In [14]:
# Combine results
all_results = pd.concat([stage1_df, pd.DataFrame(stage2)]).sort_values("val_logloss")

In [15]:
all_results

,features,learning_rate,max_depth,num_leaves,min_child_samples,n_trees,val_logloss
4,base_w_pitch,0.02,-1,127,2000,3576,0.459148
1,base_w_pitch,0.02,-1,127,1000,4140,0.459161
3,base_w_pitch,0.01,-1,127,2000,6611,0.459232
13,base_w_pitch,0.02,-1,255,2000,2179,0.459288
12,base_w_pitch,0.01,-1,255,2000,3732,0.459290
...,...,...,...,...,...,...,...
32,base,0.10,-1,127,500,154,0.504133
30,base,0.10,-1,127,100,115,0.504148
36,base,0.10,-1,255,250,97,0.504151
39,base,0.10,-1,255,2000,96,0.504162


In [16]:
# Best performing parameters
best = all_results.iloc[0]
print(best)

features             base_w_pitch
learning_rate                0.02
max_depth                      -1
num_leaves                    127
min_child_samples            2000
n_trees                      3576
val_logloss              0.459148
Name: 4, dtype: object


# Final Model

In [18]:
# Use parameters from best fit to create final model
final_model = LGBMClassifier(
    n_estimators=int(best["n_trees"] * 1.1),   # slightly more trees for ~11% more data
    learning_rate=best["learning_rate"],
    max_depth=int(best["max_depth"]),
    num_leaves=int(best["num_leaves"]),
    min_child_samples=int(best["min_child_samples"]),
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

# Fit final model
final_model.fit(df1[base_w_pitch], df1[target])

# Save model
joblib.dump(final_model, "swing_probability_model.pkl")

['swing_probability_model.pkl']